# Cell-State Reachability: a fail-closed tutorial

This notebook asks a deliberately narrow question:

> Given a target direction and a library of **supplied, measured perturbation-effect
> atoms**, what can a non-negative combination of those atoms reproduce?

It teaches the numerical contract, the infeasibility certificate, honest holdouts,
library coverage, and labeled bring-your-own-target alignment. Everything computed here
is deterministic and synthetic. The only repository result it reads is the frozen,
data-free validation-harness summary. It needs no `slice/` directory or external data.

The outputs are geometry and diagnostics—not a biological verdict, dose, recipe, gene
importance score, or wet-lab recommendation.

## Goal

By the end you will be able to:

1. build the portable `(perturbations, genes)` effect-dictionary format;
2. tell strict numerical cone membership from an application-declared cosine bar;
3. inspect a model-relative separator when a target is outside the measured cone;
4. use random-gene holdout only as a diagnostic and prefer structured/module holdout;
5. compare the cheap separator-derived acquisition order with a separately computed
   realized order; and
6. align a labeled target by gene name, fail closed on missing/incompatible labels, and
   map coefficients and signed gaps back to labels.

## Setup

Run from a repository checkout with the core requirements installed. The setup below also
works when a notebook runner chooses `tutorial/` as the kernel directory; it adds only the
detected repository root to the import path and prints no machine-specific path.

In [1]:
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import sys

import numpy as np
from scipy.stats import spearmanr

repo_root = Path.cwd()
if not (repo_root / "reachability.py").is_file():
    repo_root = repo_root.parent
if not (repo_root / "reachability.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or tutorial directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from effect_dictionary import (
    build_effect_dictionary,
    load_effect_dictionary,
    save_effect_dictionary,
    validate_effect_dictionary,
)
from library_coverage import coverage_report, rank_acquisitions
from reachability import InputError, held_out_alignment, project_cone
from validation import (
    LabeledEffects,
    LabeledTarget,
    Provenance,
    align_labeled_problem,
    grouped_gene_splits,
)

np.set_printoptions(precision=4, suppress=True)
print("Imports OK; examples are deterministic and data-free.")

Imports OK; examples are deterministic and data-free.


### Key assumptions

- Every effect row and the target use the same signed coordinate system and units.
- Coefficients are non-negative; the model does not apply a measured effect “backwards.”
- Additivity is a modeling assumption, not a demonstrated biological combination effect.
- A separator certifies separation from **this measured dictionary under this metric**. It
  does not certify biological impossibility.
- A high directional cosine does not establish magnitude accuracy, durability, function,
  donor generalization, or state conversion.

## Steps

### 1. Build a portable effect dictionary

The adapter accepts an already-preprocessed dense or SciPy sparse cell-by-gene matrix.
For each perturbation it computes the pooled condition mean minus the pooled control mean.
The control is not retained as an atom. Rows are sorted by perturbation label.

The tiny screen below is synthetic. Its names say `measured_*` to reinforce the unit being
ranked later: a perturbation-effect **profile that has already been supplied**, not an
unknown intervention whose profile the geometry could guess.

In [2]:
rng = np.random.default_rng(7)
genes = np.array(["G_A", "G_B", "G_C", "G_D", "G_E", "G_F"])
control_state = np.array([4.0, 3.5, 4.5, 2.0, 3.0, 4.0])
true_programs = {
    "ctrl": np.zeros(6),
    "measured_alpha": np.array([1.0, 0.2, 0.0, 0.0, 0.0, 0.0]),
    "measured_beta": np.array([0.0, 1.0, 0.3, 0.0, 0.0, 0.0]),
    "measured_gamma": np.array([0.0, 0.0, 0.0, 0.0, 1.0, -1.0]),
}

cell_blocks = []
condition_labels = []
for label, program in true_programs.items():
    n_cells = 16
    cell_blocks.append(
        control_state + program + rng.normal(0.0, 0.03, size=(n_cells, genes.size))
    )
    condition_labels.extend([label] * n_cells)

dictionary = build_effect_dictionary(
    np.vstack(cell_blocks),
    np.asarray(condition_labels),
    control_label="ctrl",
    gene_names=genes,
    dtype=np.float64,
)

print("format problems:", validate_effect_dictionary(dictionary) or "none")
print("E shape (perturbations, genes):", dictionary["E"].shape)
print("perturbation labels:", dictionary["perts"].tolist())
print("gene labels:", dictionary["genes"].tolist())
print("pooled cell counts (provenance only):", dictionary["ncells"].tolist())

format problems: none
E shape (perturbations, genes): (3, 6)
perturbation labels: ['measured_alpha', 'measured_beta', 'measured_gamma']
gene labels: ['G_A', 'G_B', 'G_C', 'G_D', 'G_E', 'G_F']
pooled cell counts (provenance only): [16, 16, 16]


In [3]:
# The NPZ round-trip uses Unicode/numeric arrays only; loading never enables pickle.
with TemporaryDirectory() as temporary_directory:
    portable_path = Path(temporary_directory) / "synthetic_effects.npz"
    save_effect_dictionary(portable_path, dictionary)
    restored = load_effect_dictionary(portable_path)

np.testing.assert_allclose(restored["E"], dictionary["E"])
np.testing.assert_array_equal(restored["perts"], dictionary["perts"])
print("safe NPZ round-trip: OK")

safe NPZ round-trip: OK


This adapter does **not** normalize raw counts, estimate pseudobulk effects by biological
replicate, correct batches/donors, select genes, or estimate uncertainty. `ncells` is not a
replicate count. On real data, make those scientific choices upstream; feed this helper an
additive scale on which arithmetic means and subtraction match your estimand.

### 2. Project one target and inspect the certificate

Let `E` be the effect matrix. The reachable set is

\[
\mathcal{C}(E)=\{w^\top E: w\ge 0\}.
\]

`project_cone` finds the closest point in that cone. We construct one exact non-negative
combination and one target with an added direction orthogonal to every measured atom. The
second example is intentionally outside even the atoms' linear span, so it is not a
tautological sign flip of one row.

In [4]:
effects = dictionary["E"].astype(float)
inside_target = 0.7 * effects[0] + 1.1 * effects[1]

# Construct a deterministic component orthogonal to the measured row space.
seed_direction = np.array([0.0, 0.0, 0.0, 1.0, 0.0, 0.0])
row_space_weights = np.linalg.lstsq(effects.T, seed_direction, rcond=None)[0]
gap_axis = seed_direction - row_space_weights @ effects
gap_axis /= np.linalg.norm(gap_axis)
outside_target = inside_target + 0.75 * np.linalg.norm(inside_target) * gap_axis

inside_result = project_cone(effects, inside_target)
outside_result = project_cone(effects, outside_target)

for label, result in [("exact combination", inside_result), ("outside target", outside_result)]:
    print(
        f"{label:18s} status={result.geometry_status:18s} "
        f"cosine={result.cosine:.3f} residual_fraction={result.residual_fraction:.3f}"
    )

exact combination  status=inside_tolerance   cosine=1.000 residual_fraction=0.000
outside target     status=outside_model_cone cosine=0.800 residual_fraction=0.600


In [5]:
# A certified separator has non-positive inner product with every current atom,
# but positive inner product with the outside target.
separator = outside_result.dual_separator
assert separator is not None
atom_separator_scores = effects @ separator
target_separator_score = float(outside_target @ separator)

print("maximum current-atom · separator:", f"{atom_separator_scores.max():.3e}")
print("outside target · separator:       ", f"{target_separator_score:.3f}")
print("KKT violation:                    ", f"{outside_result.kkt_violation:.3e}")
print("separation margin:                ", f"{outside_result.separation_margin:.3f}")

maximum current-atom · separator: 5.146e-16
outside target · separator:        1.181
KKT violation:                     4.441e-16
separation margin:                 0.600


The strict status answers a numerical membership question at the solver's declared
tolerance. The cosine answers a different question: how well the closest cone direction
aligns with the target. An outside target can still have a high cosine. Never relabel a
cosine threshold as exact cone membership.

### 3. Keep strict status separate from declared threshold coverage

For a target catalog, `coverage_report(..., reach_cosine=None)` counts only
`inside_tolerance`. Passing a cosine bar creates an application-declared directional
coverage rule. It does not change any target's `geometry_status`.

Here the declared bar is **0.85**. That value is a visible tutorial parameter, not a
biological truth or a threshold chosen from outcomes.

In [6]:
DECLARED_COSINE = 0.85
exact_catalog = [
    effects[0],
    0.6 * effects[0] + 1.1 * effects[1],
    0.4 * effects[2] + 0.3 * effects[1],
]
target_catalog = np.vstack(
    [
        *exact_catalog,
        exact_catalog[1] + 0.10 * np.linalg.norm(exact_catalog[1]) * gap_axis,
        exact_catalog[0] - 0.25 * np.linalg.norm(exact_catalog[0]) * gap_axis,
        exact_catalog[2] + 0.60 * np.linalg.norm(exact_catalog[2]) * gap_axis,
        exact_catalog[1] + 1.50 * np.linalg.norm(exact_catalog[1]) * gap_axis,
    ]
)
target_labels = [f"catalog_{index}" for index in range(target_catalog.shape[0])]

strict_coverage = coverage_report(effects, target_catalog)
declared_coverage = coverage_report(
    effects, target_catalog, reach_cosine=DECLARED_COSINE
)

print("target     strict geometry       cosine  strict?  cosine>=0.85?")
for index, label in enumerate(target_labels):
    print(
        f"{label:10s} {strict_coverage.statuses[index]:20s} "
        f"{strict_coverage.cosines[index]:7.3f} "
        f"{str(bool(strict_coverage.reachable[index])):>8s} "
        f"{str(bool(declared_coverage.reachable[index])):>14s}"
    )
print()
print("strict inside-cone fraction:    ", f"{strict_coverage.reachable_fraction:.3f}")
print("declared cosine coverage (0.85):", f"{declared_coverage.reachable_fraction:.3f}")

target     strict geometry       cosine  strict?  cosine>=0.85?
catalog_0  inside_tolerance       1.000     True           True
catalog_1  inside_tolerance       1.000     True           True
catalog_2  inside_tolerance       1.000     True           True
catalog_3  outside_model_cone     0.995    False           True
catalog_4  outside_model_cone     0.970    False           True
catalog_5  outside_model_cone     0.857    False           True
catalog_6  outside_model_cone     0.555    False          False

strict inside-cone fraction:     0.429
declared cosine coverage (0.85): 0.857


The two fractions intentionally differ. Report the rule beside the number: “strict
inside-cone fraction” or “fraction with best cosine ≥ 0.85.” The unqualified word
“reachable” is ambiguous here.

### 4. Treat random-gene holdout as a diagnostic, not evidence

`held_out_alignment` fits coefficients on one coordinate set and scores the frozen
coefficients on another. A random half-gene split catches some overfitting, but correlated
genes from the same module can land on both sides. It therefore does not create independent
replicates or validate structured specificity.

The bounded example below contains module-correlated genes plus a strong common response.
We compare the cone with a one-atom common-response baseline. Positive “specific gain”
means the full cone beats that nuisance baseline. We average eight deterministic random
splits and eight splits that keep whole modules together.

In [7]:
rng = np.random.default_rng(0)
n_modules, genes_per_module, n_atoms = 12, 6, 7
n_structured_genes = n_modules * genes_per_module
module_ids = np.repeat(np.arange(n_modules), genes_per_module)

common = np.repeat(rng.normal(size=n_modules), genes_per_module)
common += rng.normal(scale=0.15, size=n_structured_genes)
common /= np.std(common)

atom_modules = rng.normal(size=(n_atoms, n_modules))
atom_modules -= atom_modules.mean(axis=0, keepdims=True)
specific = np.repeat(atom_modules, genes_per_module, axis=1)
specific += rng.normal(scale=0.05, size=(n_atoms, n_structured_genes))
structured_effects = 2.0 * common + specific

nuisance = np.repeat(rng.normal(size=n_modules), genes_per_module)
structured_null_target = 2.0 * common + 1.2 * nuisance
structured_null_target += rng.normal(scale=0.10, size=n_structured_genes)

def held_out_specificity(fit_indices, score_indices):
    cone = held_out_alignment(
        structured_effects, structured_null_target, fit_indices, score_indices
    ).held_out_cosine
    common_only = held_out_alignment(
        structured_effects.mean(axis=0, keepdims=True),
        structured_null_target,
        fit_indices,
        score_indices,
    ).held_out_cosine
    return cone, common_only, cone - common_only

random_diagnostics = []
for split_seed in range(8):
    order = np.random.default_rng(100 + split_seed).permutation(n_structured_genes)
    random_diagnostics.append(
        held_out_specificity(
            np.sort(order[: n_structured_genes // 2]),
            np.sort(order[n_structured_genes // 2 :]),
        )
    )

module_labels = tuple(f"module_{module}" for module in module_ids)
module_diagnostics = [
    held_out_specificity(fit_indices, score_indices)
    for fit_indices, score_indices in grouped_gene_splits(
        module_labels, n_splits=8, seed=200
    )
]

random_mean = np.mean(random_diagnostics, axis=0)
module_mean = np.mean(module_diagnostics, axis=0)
print("split rule       cone cosine  common baseline  specific gain")
print(f"random genes       {random_mean[0]:8.3f}       {random_mean[1]:8.3f}      {random_mean[2]:+8.3f}")
print(f"whole modules      {module_mean[0]:8.3f}       {module_mean[1]:8.3f}      {module_mean[2]:+8.3f}")

split rule       cone cosine  common baseline  specific gain
random genes          0.925          0.897        +0.028
whole modules         0.881          0.889        -0.008


The high raw cosine is largely a common-response success mode. Random-gene holdout still
suggests a small positive specific gain because module mates leak across the split;
whole-module holdout removes that apparent gain in this null scenario.

This is a diagnostic illustration, not a new inference. The repository's frozen
adversarial harness repeats the structured challenge 120 times with independent random
streams. We read that result rather than rerunning or rewriting it.

In [8]:
with (repo_root / "results" / "validation_harness.json").open(encoding="utf-8") as handle:
    frozen_harness = json.load(handle)

structured = frozen_harness["scenarios"]["structured_specificity_calibration"]
print("frozen harness status:", frozen_harness["status"])
print("scope:", frozen_harness["scope"])
print("structured trials:", structured["trials"])
print("null raw random-gene cosine mean:      ", f"{structured['null_raw_random_gene_cosine_mean']:.3f}")
print("null random-gene specific gain mean:   ", f"{structured['null_random_gene_improvement_mean']:+.3f}")
print("null module-holdout specific gain mean:", f"{structured['null_module_holdout_improvement_mean']:+.3f}")
print("true-alt module-holdout gain mean:     ", f"{structured['alternative_module_holdout_improvement_mean']:+.3f}")

frozen harness status: PASS
scope: deterministic synthetic software/statistical contract; not biological validation
structured trials: 120
null raw random-gene cosine mean:       0.889
null random-gene specific gain mean:    +0.028
null module-holdout specific gain mean: -0.024
true-alt module-holdout gain mean:      +0.043


No random-gene-shuffle p-value is promoted here. A defensible analysis needs a holdout or
null that respects the exchangeable unit and the correlation structure, plus synchronized
multiplicity control when several hypotheses are tested. Random-gene splits remain useful
for debugging stability; they are not a substitute for donor, guide, source, module, or
other scientifically structured holdout.

### 5. Test the separator-derived acquisition ranking against realized coverage

`rank_acquisitions` takes a starting library, target catalog, and a matrix of candidate
effect atoms. Those candidates must already be **supplied measured profiles in the same
coordinate system**. The function cannot score an unknown perturbation without an effect
vector and does not recommend a wet-lab intervention.

- `order` is always the cheap **certificate-score order**.
- `realized_order` exists only when `realized=True`; it comes from actually appending each
  candidate and recomputing mean catalog cosine.

The separator is certified; the resulting candidate ranking is a heuristic. Keeping the
arrays separate prevents a tautological “certificate recovered itself” check.
The small retrospective example below holds six synthetic measured atoms outside a
three-atom starting library. It is intentionally not perfect: the top pick agrees, but two
middle ranks swap.

In [9]:
rng = np.random.default_rng(0)
starting_library = rng.normal(size=(3, 6))
supplied_candidates = rng.normal(size=(6, 6))
all_measured_atoms = np.vstack([starting_library, supplied_candidates])

# A bounded synthetic catalog generated before either ranking is inspected.
catalog_weights = rng.uniform(size=(9, 9))
catalog_weights[catalog_weights < 0.6] = 0.0
acquisition_catalog = catalog_weights @ all_measured_atoms
acquisition_catalog += rng.normal(scale=0.2, size=acquisition_catalog.shape)
candidate_labels = np.array([f"measured_candidate_{index}" for index in range(6)])

acquisition = rank_acquisitions(
    starting_library,
    acquisition_catalog,
    supplied_candidates,
    realized=True,
)
rho = float(spearmanr(acquisition.certificate_score, acquisition.realized_gain).statistic)
certificate_rank = np.empty(candidate_labels.size, dtype=int)
certificate_rank[acquisition.order] = np.arange(1, candidate_labels.size + 1)
realized_rank = np.empty(candidate_labels.size, dtype=int)
realized_rank[acquisition.realized_order] = np.arange(1, candidate_labels.size + 1)

print("candidate              cert rank  realized rank  cert score  realized gain")
for index in acquisition.order:  # explicitly the certificate order
    print(
        f"{candidate_labels[index]:24s} {certificate_rank[index]:9d} "
        f"{realized_rank[index]:14d} {acquisition.certificate_score[index]:11.3f} "
        f"{acquisition.realized_gain[index]:14.3f}"
    )
print()
print("Spearman(certificate, realized):", f"{rho:.3f}")
print("same top measured atom:         ", bool(acquisition.order[0] == acquisition.realized_order[0]))
print("identical full order:           ", bool(np.array_equal(acquisition.order, acquisition.realized_order)))

candidate              cert rank  realized rank  cert score  realized gain
measured_candidate_0             1              1       2.802          0.127
measured_candidate_4             2              3       2.436          0.084
measured_candidate_3             3              2       1.957          0.104
measured_candidate_2             4              4       1.556          0.082
measured_candidate_5             5              5       0.801          0.040
measured_candidate_1             6              6       0.093          0.001

Spearman(certificate, realized): 0.943
same top measured atom:          True
identical full order:            False


The certificate score is an approximation to a separately computed retrospective
comparator, not a predictor of experimental outcome. A real deployment would compute only the cheap order; a benchmark
with measured held-out atoms can request the expensive sweep to test whether the
approximation deserves trust in that setting.

### 6. Bring your own target: align labels and fail closed

Positional matching is unsafe. `align_labeled_problem` validates provenance, label
uniqueness, units, orientations, finiteness, and gene coverage, then reorders the target by
gene name. Missing genes fail unless intersection is explicitly enabled with declared
minimum coverage gates.

The hashes below are synthetic placeholders for this tutorial. In real work, record actual
source hashes and honest units/context metadata.

In [10]:
effects_provenance = Provenance(
    dataset_id="tutorial-effects-v1",
    source_sha256="0" * 64,
    gene_namespace="tutorial-gene-symbol",
    units="mean_log_expression_change",
    modality="synthetic measured effects",
    context="tutorial",
    timepoint="single synthetic timepoint",
    orientation="perturbations_by_genes",
)
target_provenance = replace(
    effects_provenance,
    dataset_id="tutorial-target-v1",
    source_sha256="1" * 64,
    modality="synthetic target direction",
    orientation="genes",
)

labeled_effects = LabeledEffects(
    values=effects,
    perturbations=tuple(dictionary["perts"].tolist()),
    genes=tuple(dictionary["genes"].tolist()),
    provenance=effects_provenance,
)

# Deliberately scramble the target axis. Alignment must use names, not positions.
target_permutation = np.array([2, 5, 0, 4, 1, 3])
labeled_target = LabeledTarget(
    values=outside_target[target_permutation],
    genes=tuple(dictionary["genes"][target_permutation].tolist()),
    provenance=target_provenance,
)
aligned = align_labeled_problem(labeled_effects, labeled_target)
labeled_result = project_cone(aligned.effects, aligned.target)

active_rows = [
    (label, float(coefficient))
    for label, coefficient in zip(aligned.perturbations, labeled_result.coefficients)
    if coefficient > 1e-10
]
active_rows.sort(key=lambda item: -item[1])

# Preserve the residual sign: target - fitted. Rank by magnitude only for display.
gap_order = np.argsort(-np.abs(labeled_result.residual))[:4]
signed_gap_rows = [
    (aligned.genes[index], float(labeled_result.residual[index]))
    for index in gap_order
]

print("aligned gene order:", aligned.genes)
print("effect/target gene coverage:", aligned.effects_gene_fraction, aligned.target_gene_fraction)
print("geometry status:", labeled_result.geometry_status)
print("active measured atoms (label, coefficient):", active_rows)
print("largest signed gaps (gene, target - fitted):")
for gene, signed_gap in signed_gap_rows:
    direction = "higher than fitted" if signed_gap > 0 else "lower than fitted"
    print(f"  {gene:4s} {signed_gap:+.4f}  ({direction})")

aligned gene order: ('G_A', 'G_B', 'G_C', 'G_D', 'G_E', 'G_F')
effect/target gene coverage: 1.0 1.0
geometry status: outside_model_cone
active measured atoms (label, coefficient): [('measured_beta', 1.0999999999999996), ('measured_alpha', 0.7000000000000001)]
largest signed gaps (gene, target - fitted):
  G_D  +1.0865  (higher than fitted)
  G_E  -0.0069  (lower than fitted)
  G_F  +0.0069  (higher than fitted)
  G_B  -0.0043  (lower than fitted)


In [11]:
# Removing one target label fails closed by default instead of silently intersecting.
missing_gene_target = replace(
    labeled_target,
    values=labeled_target.values[:-1],
    genes=labeled_target.genes[:-1],
)
try:
    align_labeled_problem(labeled_effects, missing_gene_target)
except InputError as error:
    print("expected fail-closed alignment:", error)

expected fail-closed alignment: gene sets differ; intersection must be explicitly enabled


The coefficient table is keyed by perturbation labels, not row numbers. The gap table
keeps the sign of `target - fitted`; taking absolute values to rank display rows must not
erase whether the target asks for a coordinate above or below the fitted direction.

Neither table is a treatment prescription. Coefficients describe the model-relative
projection of supplied measured atoms, and signed gaps describe coordinates the current
dictionary did not reproduce.

## Checks

These executable checks bind the tutorial prose to the actual outputs and API semantics.

In [12]:
assert validate_effect_dictionary(restored) == []
assert inside_result.geometry_status == "inside_tolerance"
assert outside_result.geometry_status == "outside_model_cone"
assert atom_separator_scores.max() <= 1e-8 and target_separator_score > 0
assert strict_coverage.reachable_fraction < declared_coverage.reachable_fraction
np.testing.assert_array_equal(
    acquisition.order,
    np.argsort(-acquisition.certificate_score, kind="stable"),
)
np.testing.assert_array_equal(
    acquisition.realized_order,
    np.argsort(-acquisition.realized_gain, kind="stable"),
)
np.testing.assert_allclose(aligned.target, outside_target)
assert frozen_harness["status"] == "PASS"
print("All tutorial checks passed.")

All tutorial checks passed.


## Next Steps

For your own analysis:

1. normalize and estimate perturbation effects at the appropriate biological-replicate
   level before building the dictionary;
2. record honest gene namespace, units, source hashes, modality, context, timepoint, and
   axis orientation for both effects and target;
3. use `align_labeled_problem` and require exact gene sets unless an explicit intersection
   and minimum-coverage gate were prespecified;
4. choose structured held-out units that match the claim—modules, donors, guides, sources,
   timepoints, or studies—not a random-gene shuffle promoted as inference;
5. report strict cone status separately from any declared cosine-threshold coverage; and
6. if evaluating acquisition, benchmark the certificate order against a separately
   computed realized order on supplied measured held-out atoms.

The repository's frozen biological case studies remain context and stress tests. They do
not turn this notebook's synthetic geometry into evidence of state conversion, intervention
efficacy, donor generalization, or magnitude accuracy.